# Step 1 & 2: English-Bhojpuri Dataset Inspection & Cleaning
**Hackathon Role**: Member 1 (ML / NLP Workflow)  
**Dataset Source**: `nilayshenai/English-Bhojpuri_Translation_Dataset`  
**Goal**: Inspect parallel corpus columns, analyze missing/duplicate entries, compute length distributions, and verify alignment.

In [ ]:
import json
import os
import pandas as pd

# Find dataset path
raw_path = os.path.join('..', 'data', 'raw', 'engbhoj.jsonl')
if not os.path.exists(raw_path):
    raw_path = os.path.join('data', 'raw', 'engbhoj.jsonl')

print(f"Raw dataset path: {raw_path} (exists: {os.path.exists(raw_path)})")

## 1. Load Raw JSON Lines Data

In [ ]:
records = []
with open(raw_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        t = obj.get('translation', {})
        records.append({'en': t.get('en', ''), 'bho': t.get('bho', '')})

df = pd.DataFrame(records)
print(f"Loaded total raw pairs: {len(df)}")
df.head()

## 2. Check for Missing, Null, and Blank Entries

In [ ]:
null_en = df['en'].isna().sum()
null_bho = df['bho'].isna().sum()
blank_en = (df['en'].astype(str).str.strip() == '').sum()
blank_bho = (df['bho'].astype(str).str.strip() == '').sum()

print(f"Null English: {null_en} | Blank English: {blank_en}")
print(f"Null Bhojpuri: {null_bho} | Blank Bhojpuri: {blank_bho}")

## 3. Duplicate Analysis and Removal

In [ ]:
exact_duplicates = df.duplicated(subset=['en', 'bho']).sum()
print(f"Exact duplicate sentence pairs: {exact_duplicates}")

# Remove exact duplicates
df_dedup = df.drop_duplicates(subset=['en', 'bho']).copy()
print(f"Dataset size after removing duplicates: {len(df_dedup)}")

## 4. Sequence Length & Token Statistics

In [ ]:
df_dedup['en_words'] = df_dedup['en'].apply(lambda x: len(str(x).split()))
df_dedup['bho_words'] = df_dedup['bho'].apply(lambda x: len(str(x).split()))
df_dedup['en_chars'] = df_dedup['en'].apply(lambda x: len(str(x)))
df_dedup['bho_chars'] = df_dedup['bho'].apply(lambda x: len(str(x)))

df_dedup[['en_words', 'bho_words', 'en_chars', 'bho_chars']].describe(percentiles=[0.5, 0.9, 0.95, 0.99])

## 5. Sample Inspection

In [ ]:
for i, (_, row) in enumerate(df_dedup.head(10).iterrows(), 1):
    print(f"{i}. EN:  {row['en']}")
    print(f"   BHO: {row['bho']}")
    print()
